In [1]:
library(reticulate)
reticulate::py_install(c("pandas", "numpy", "scipy"), pip = TRUE)

ERROR: Error: Tools for managing Python virtual environments are not installed.

Install venv with:
$ sudo apt-get install python3-venv python3-pip python3-dev




In [ ]:

suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
  library(data.table)
  library(CellEnergy)
})

parse_named_args <- function(args) {
  out <- list()
  if (length(args) == 0) return(out)
  i <- 1
  while (i <= length(args)) {
    key <- args[[i]]
    if (grepl("^--", key)) {
      key <- sub("^--", "", key)
      if (i + 1 <= length(args) && !grepl("^--", args[[i + 1]])) {
        out[[key]] <- args[[i + 1]]
        i <- i + 2
      } else {
        out[[key]] <- TRUE
        i <- i + 1
      }
    } else {
      i <- i + 1
    }
  }
  out
}

args <- parse_named_args(commandArgs(trailingOnly = TRUE))

expr_path <- if (!is.null(args$expr)) args$expr else "/path/GSE67835/GSE67835_merged_out/GSE67835_expression_cell_by_gene_raw_counts.csv"
soft_path <- if (!is.null(args$soft)) args$soft else "/path/GSE67835/GSE67835_merged_out/GSE67835_family.soft"
out_dir   <- if (!is.null(args$out))  args$out  else "/path/GSE67835/Seurat_CellEnergy_full_pipeline"

min_cells <- if (!is.null(args$min_cells)) as.integer(args$min_cells) else 3
min_features <- if (!is.null(args$min_features)) as.integer(args$min_features) else 200
n_hvg <- if (!is.null(args$n_hvg)) as.integer(args$n_hvg) else 2000

dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

cat("[INFO] Expression matrix path:\n", expr_path, "\n", sep = "")
cat("[INFO] SOFT metadata path:\n", soft_path, "\n", sep = "")
cat("[INFO] Output directory:\n", out_dir, "\n\n", sep = "")

if (!file.exists(expr_path)) stop("[ERROR] Expression matrix file does not exist: ", expr_path)
if (!file.exists(soft_path)) stop("[ERROR] SOFT metadata file does not exist: ", soft_path)

trim_string <- function(x) {
  gsub("^\\s+|\\s+$", "", x)
}

sanitize_key <- function(x) {
  x <- tolower(trim_string(x))
  x <- gsub("[^a-z0-9]+", "_", x)
  x <- gsub("^_+|_+$", "", x)
  x
}

extract_gsm <- function(x) {
  x <- as.character(x)
  m <- regexpr("GSM[0-9]+", x)
  out <- x
  hit <- m > 0
  matches <- regmatches(x, m)
  out[hit] <- matches
  out
}

write_matrix_with_id <- function(mat, file, id_col = "ID") {
  df <- as.data.frame(as.matrix(mat), check.names = FALSE)
  df <- cbind(
    setNames(data.frame(rownames(df), check.names = FALSE), id_col),
    df
  )
  fwrite(df, file = file)
}

minmax_scale_columns <- function(mat) {
  mat <- as.matrix(mat)
  col_min <- apply(mat, 2, min, na.rm = TRUE)
  col_max <- apply(mat, 2, max, na.rm = TRUE)
  col_range <- col_max - col_min
  col_range[col_range == 0 | is.na(col_range)] <- 1
  scaled <- sweep(mat, 2, col_min, FUN = "-")
  scaled <- sweep(scaled, 2, col_range, FUN = "/")
  scaled[is.na(scaled)] <- 0
  scaled
}

get_assay_data_compat <- function(obj, assay = "RNA", layer_name = "counts", slot_name = "counts") {
  DefaultAssay(obj) <- assay
  tryCatch(
    GetAssayData(obj, assay = assay, layer = layer_name),
    error = function(e) {
      GetAssayData(obj, assay = assay, slot = slot_name)
    }
  )
}

parse_soft_metadata <- function(path) {
  lines <- readLines(path, warn = FALSE)
  samples <- list()
  current <- list()
  sample_i <- 0

  flush_current <- function(x) {
    if (length(x) == 0) return(NULL)
    if (is.null(x$GSM) && !is.null(x$sample_accession)) x$GSM <- x$sample_accession
    if (is.null(x$Cell) && !is.null(x$GSM)) x$Cell <- x$GSM
    x
  }

  for (ln in lines) {
    if (grepl("^\\^SAMPLE\\s*=", ln)) {
      old <- flush_current(current)
      if (!is.null(old)) samples[[length(samples) + 1]] <- old
      sample_i <- sample_i + 1
      current <- list(
        soft_sample = trim_string(sub("^\\^SAMPLE\\s*=\\s*", "", ln))
      )
      next
    }

    if (length(current) == 0) next

    if (grepl("^!Sample_geo_accession\\s*=", ln)) {
      current$GSM <- trim_string(sub("^!Sample_geo_accession\\s*=\\s*", "", ln))
      current$Cell <- current$GSM
      next
    }

    if (grepl("^!Sample_title\\s*=", ln)) {
      current$title <- trim_string(sub("^!Sample_title\\s*=\\s*", "", ln))
      next
    }

    if (grepl("^!Sample_source_name_ch1\\s*=", ln)) {
      current$source_name <- trim_string(sub("^!Sample_source_name_ch1\\s*=\\s*", "", ln))
      next
    }

    if (grepl("^!Sample_characteristics_ch1\\s*=", ln)) {
      val <- trim_string(sub("^!Sample_characteristics_ch1\\s*=\\s*", "", ln))
      if (grepl(":", val)) {
        key <- sanitize_key(sub(":.*$", "", val))
        value <- trim_string(sub("^[^:]+:\\s*", "", val))
        if (!is.null(current[[key]])) {
          current[[key]] <- paste(current[[key]], value, sep = "; ")
        } else {
          current[[key]] <- value
        }
      } else {
        if (!is.null(current$characteristics_ch1)) {
          current$characteristics_ch1 <- paste(current$characteristics_ch1, val, sep = "; ")
        } else {
          current$characteristics_ch1 <- val
        }
      }
      next
    }

    if (grepl("^!Sample_description\\s*=", ln)) {
      current$description <- trim_string(sub("^!Sample_description\\s*=\\s*", "", ln))
      next
    }

    if (grepl("^!Sample_supplementary_file", ln)) {
      val <- trim_string(sub("^!Sample_supplementary_file[^=]*=\\s*", "", ln))
      val_base <- basename(val)
      if (!is.null(current$supplementary_file)) {
        current$supplementary_file <- paste(current$supplementary_file, val_base, sep = "; ")
      } else {
        current$supplementary_file <- val_base
      }
      next
    }
  }

  old <- flush_current(current)
  if (!is.null(old)) samples[[length(samples) + 1]] <- old

  if (length(samples) == 0) stop("[ERROR] No sample metadata parsed from SOFT file.")

  all_cols <- unique(unlist(lapply(samples, names)))
  metadata <- as.data.frame(
    setNames(
      lapply(all_cols, function(cc) {
        vapply(samples, function(s) {
          if (!is.null(s[[cc]])) as.character(s[[cc]]) else NA_character_
        }, character(1))
      }),
      all_cols
    ),
    stringsAsFactors = FALSE,
    check.names = FALSE
  )

  if (!"GSM" %in% colnames(metadata)) {
    stop("[ERROR] Cannot find GSM accession in SOFT metadata.")
  }

  metadata$Cell <- metadata$GSM
  rownames(metadata) <- metadata$Cell
  metadata
}

read_expression_cell_by_gene <- function(path) {
  expr_df <- fread(path, data.table = FALSE, check.names = FALSE)
  cat("[INFO] Raw expression table dimension from CSV:\n")
  print(dim(expr_df))
  cat("[INFO] First columns:\n")
  print(head(colnames(expr_df), 10))

  known_cell_cols <- c("Cell", "cell", "GSM", "gsm", "Sample", "sample", "ID", "id", "cell_id", "CellID")
  hit <- intersect(known_cell_cols, colnames(expr_df))

  if (length(hit) > 0) {
    cell_col <- hit[1]
  } else {
    first_col <- expr_df[[1]]
    first_num <- suppressWarnings(as.numeric(first_col))
    if (any(is.na(first_num)) || !is.numeric(expr_df[[1]])) {
      cell_col <- colnames(expr_df)[1]
    } else {
      stop("[ERROR] Cannot identify cell ID column. Please rename the first column to Cell.")
    }
  }

  cat("[INFO] Cell ID column used in expression matrix:\n")
  print(cell_col)

  cell_idx <- match(cell_col, colnames(expr_df))
  cells_raw <- as.character(expr_df[[cell_idx]])
  feature_df <- expr_df[, -cell_idx, drop = FALSE]

  # Remove duplicated or empty feature names before matrix conversion
  feature_names <- colnames(feature_df)
  feature_names[is.na(feature_names) | feature_names == ""] <- paste0("unknown_feature_", which(is.na(feature_names) | feature_names == ""))
  colnames(feature_df) <- make.unique(feature_names)

  mat <- as.matrix(feature_df)
  suppressWarnings(storage.mode(mat) <- "numeric")
  mat[is.na(mat)] <- 0
  rownames(mat) <- cells_raw

  # HTSeq summary rows/columns are not genes and must be removed before Seurat.
  htseq_summary_features <- c(
    "__no_feature", "__ambiguous", "__too_low_aQual", "__not_aligned", "__alignment_not_unique",
    "no_feature", "ambiguous", "too_low_aQual", "not_aligned", "alignment_not_unique"
  )
  bad_features <- intersect(htseq_summary_features, colnames(mat))
  if (length(bad_features) > 0) {
    cat("[INFO] Removing HTSeq summary features, not real genes:\n")
    print(bad_features)
    mat <- mat[, setdiff(colnames(mat), bad_features), drop = FALSE]
  }

  # Convert cells x genes to genes x cells for Seurat.
  counts <- t(mat)
  counts <- as(counts, "dgCMatrix")
  counts
}


counts <- read_expression_cell_by_gene(expr_path)
metadata <- parse_soft_metadata(soft_path)

cat("[INFO] Parsed SOFT metadata dimension:\n")
print(dim(metadata))
cat("[INFO] Parsed metadata columns:\n")
print(colnames(metadata))

# GSE67835 stores cell type in Sample_characteristics_ch1 as "cell type: ...".
# The parser converts this field to column name "cell_type".
if ("cell_type" %in% colnames(metadata)) {
  label_col <- "cell_type"
} else {
  candidate_label_cols <- c(
    "celltype", "cell_type", "CellType", "cell.type", "annotation", "Annotation",
    "label", "Label", "cluster", "Cluster", "seurat_clusters"
  )
  label_col <- intersect(candidate_label_cols, colnames(metadata))[1]
}

if (is.na(label_col) || length(label_col) == 0) {
  cat("[ERROR] Cannot automatically find cell label column.\n")
  cat("[INFO] Available metadata columns:\n")
  print(colnames(metadata))
  stop("Please manually set label_col after checking metadata.")
}

cat("[INFO] Cell label column used:\n")
print(label_col)
cat("[INFO] Cell label distribution before QC:\n")
print(table(metadata[[label_col]], useNA = "ifany"))

cat("[INFO] Count matrix before alignment, genes x cells:\n")
print(dim(counts))
cat("[INFO] Example genes:\n")
print(head(rownames(counts)))
cat("[INFO] Example cells before GSM cleanup:\n")
print(head(colnames(counts)))

# If expression cell names contain filenames such as GSM1657871_xxx.csv.gz,
# reduce them to GSM accessions for alignment with SOFT metadata.
original_cells <- colnames(counts)
gsm_cells <- extract_gsm(original_cells)
if (length(unique(gsm_cells)) == length(gsm_cells)) {
  overlap_raw <- sum(original_cells %in% rownames(metadata))
  overlap_gsm <- sum(gsm_cells %in% rownames(metadata))
  cat("[INFO] Overlap with metadata using raw cell names:\n")
  print(overlap_raw)
  cat("[INFO] Overlap with metadata using extracted GSM IDs:\n")
  print(overlap_gsm)
  if (overlap_gsm >= overlap_raw) {
    colnames(counts) <- gsm_cells
    cat("[INFO] Using GSM IDs as cell names.\n")
  }
} else {
  cat("[WARN] Extracted GSM IDs are not unique. Keeping original cell names.\n")
}


common_cells <- intersect(colnames(counts), rownames(metadata))
cat("[INFO] Number of common cells between expression and metadata:\n")
print(length(common_cells))

if (length(common_cells) == 0) {
  stop("[ERROR] No common cell IDs between expression matrix and SOFT metadata.")
}

counts <- counts[, common_cells, drop = FALSE]
metadata <- metadata[common_cells, , drop = FALSE]

cat("[INFO] Counts after alignment, genes x cells:\n")
print(dim(counts))
cat("[INFO] Metadata after alignment:\n")
print(dim(metadata))
cat("[INFO] Cell label distribution after alignment before QC:\n")
print(table(metadata[[label_col]], useNA = "ifany"))


full_expr_gene_by_cell_path <- file.path(out_dir, "full_raw_counts_gene_by_cell.rds")
full_expr_cell_by_gene_path <- file.path(out_dir, "full_raw_counts_cell_by_gene.rds")
metadata_aligned_path <- file.path(out_dir, "metadata_aligned.csv")
cell_label_path <- file.path(out_dir, "cell_labels.csv")

saveRDS(counts, file = full_expr_gene_by_cell_path)
saveRDS(t(counts), file = full_expr_cell_by_gene_path)

metadata_out <- cbind(
  Cell = rownames(metadata),
  metadata[, setdiff(colnames(metadata), "Cell"), drop = FALSE]
)
fwrite(metadata_out, file = metadata_aligned_path)

cell_label_df <- data.frame(
  Cell = rownames(metadata),
  Label = metadata[[label_col]],
  stringsAsFactors = FALSE
)
fwrite(cell_label_df, file = cell_label_path)

cat("[OK] Full raw count matrix saved, genes x cells:\n", full_expr_gene_by_cell_path, "\n", sep = "")
cat("[OK] Full raw count matrix saved, cells x genes:\n", full_expr_cell_by_gene_path, "\n", sep = "")
cat("[OK] Metadata saved:\n", metadata_aligned_path, "\n", sep = "")
cat("[OK] Cell labels saved:\n", cell_label_path, "\n", sep = "")

seurat_obj <- CreateSeuratObject(
  counts = counts,
  project = "GSE67835",
  meta.data = metadata,
  min.cells = min_cells,
  min.features = min_features
)

cat("[INFO] Seurat object after QC:\n")
print(seurat_obj)

qc_summary <- data.frame(
  n_genes_before_qc = nrow(counts),
  n_cells_before_qc = ncol(counts),
  n_genes_after_qc = nrow(seurat_obj),
  n_cells_after_qc = ncol(seurat_obj),
  min_cells = min_cells,
  min_features = min_features
)
fwrite(qc_summary, file = file.path(out_dir, "QC_summary.csv"))


seurat_obj <- NormalizeData(
  seurat_obj,
  normalization.method = "LogNormalize",
  scale.factor = 10000
)

n_hvg_use <- min(n_hvg, nrow(seurat_obj))
seurat_obj <- FindVariableFeatures(
  seurat_obj,
  selection.method = "vst",
  nfeatures = n_hvg_use
)

hvg_genes <- VariableFeatures(seurat_obj)
cat("[INFO] Number of HVGs:\n")
print(length(hvg_genes))

hvg_path <- file.path(out_dir, paste0("HVG_", length(hvg_genes), "_gene_list.csv"))
fwrite(data.frame(HVG = hvg_genes), file = hvg_path)
cat("[OK] HVG list saved:\n", hvg_path, "\n", sep = "")

# This only performs centering/scaling. No vars.to.regress is used.
seurat_obj <- ScaleData(
  seurat_obj,
  features = hvg_genes
)

raw_hvg_counts_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "counts",
  slot_name = "counts"
)[hvg_genes, ]

normalized_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "data",
  slot_name = "data"
)[hvg_genes, ]

scaled_hvg_gene_by_cell <- get_assay_data_compat(
  seurat_obj,
  assay = "RNA",
  layer_name = "scale.data",
  slot_name = "scale.data"
)[hvg_genes, ]

cat("[INFO] Raw HVG counts, genes x cells:\n")
print(dim(raw_hvg_counts_gene_by_cell))
cat("[INFO] Normalized HVG expression, genes x cells:\n")
print(dim(normalized_hvg_gene_by_cell))
cat("[INFO] Scaled HVG expression, genes x cells:\n")
print(dim(scaled_hvg_gene_by_cell))

raw_hvg_gene_by_cell_path <- file.path(out_dir, "raw_HVG_counts_gene_by_cell.csv")
normalized_hvg_gene_by_cell_path <- file.path(out_dir, "normalized_HVG_expr_gene_by_cell.csv")
scaled_hvg_gene_by_cell_path <- file.path(out_dir, "scaled_HVG_expr_gene_by_cell_for_CellEnergy.csv")
raw_hvg_cell_by_gene_path <- file.path(out_dir, "raw_HVG_counts_cell_by_gene.csv")
normalized_hvg_cell_by_gene_path <- file.path(out_dir, "normalized_HVG_expr_cell_by_gene.csv")
scaled_hvg_cell_by_gene_path <- file.path(out_dir, "scaled_HVG_expr_cell_by_gene.csv")

write_matrix_with_id(raw_hvg_counts_gene_by_cell, raw_hvg_gene_by_cell_path, id_col = "Gene")
write_matrix_with_id(normalized_hvg_gene_by_cell, normalized_hvg_gene_by_cell_path, id_col = "Gene")

# CellEnergy input: genes x cells, keep row.names in CSV.
write.csv(
  as.data.frame(as.matrix(scaled_hvg_gene_by_cell), check.names = FALSE),
  file = scaled_hvg_gene_by_cell_path,
  quote = FALSE,
  row.names = TRUE
)

write_matrix_with_id(t(as.matrix(raw_hvg_counts_gene_by_cell)), raw_hvg_cell_by_gene_path, id_col = "Cell")
write_matrix_with_id(t(as.matrix(normalized_hvg_gene_by_cell)), normalized_hvg_cell_by_gene_path, id_col = "Cell")
write_matrix_with_id(t(as.matrix(scaled_hvg_gene_by_cell)), scaled_hvg_cell_by_gene_path, id_col = "Cell")

cat("[OK] HVG matrices saved.\n")


if (requireNamespace("reticulate", quietly = TRUE)) {
  if (!reticulate::py_module_available("pandas") ||
      !reticulate::py_module_available("numpy") ||
      !reticulate::py_module_available("scipy")) {
    cat("[WARN] Python modules pandas/numpy/scipy may be missing for CellEnergy.\n")
    cat("[WARN] If calcGEn fails, run in R: reticulate::py_install(c('pandas','numpy','scipy'), pip = TRUE)\n")
  }
}

cat("[INFO] Running CellEnergy calcGEn...\n")
result <- calcGEn(
  scaled_hvg_gene_by_cell_path,
  verbose = TRUE
)

scEn <- result$scEn
GLNE <- result$GLNE

cat("[INFO] scEnergy preview:\n")
print(head(scEn))
cat("[INFO] GLNE dimension before orientation check:\n")
print(dim(GLNE))

energy_path <- file.path(out_dir, "Energy_scEn.csv")
glne_gene_by_cell_path <- file.path(out_dir, "GLNE_gene_by_cell.csv")

scEn_df <- as.data.frame(scEn, check.names = FALSE)
if (is.null(rownames(scEn_df)) || any(rownames(scEn_df) == "")) {
  rownames(scEn_df) <- colnames(scaled_hvg_gene_by_cell)
}
scEn_df <- cbind(Cell = rownames(scEn_df), scEn_df)
fwrite(scEn_df, file = energy_path)
write_matrix_with_id(GLNE, glne_gene_by_cell_path, id_col = "Gene")

cat("[OK] Single-cell energy scores saved:\n", energy_path, "\n", sep = "")
cat("[OK] GLNE gene x cell matrix saved:\n", glne_gene_by_cell_path, "\n", sep = "")


GLNE_mat <- as.matrix(GLNE)
raw_hvg_cell_by_gene <- t(as.matrix(raw_hvg_counts_gene_by_cell))
scaled_hvg_cell_by_gene <- t(as.matrix(scaled_hvg_gene_by_cell))

glne_overlap_cols_cells <- sum(colnames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))
glne_overlap_rows_cells <- sum(rownames(GLNE_mat) %in% rownames(raw_hvg_cell_by_gene))

cat("[INFO] GLNE overlap with cells in columns:\n")
print(glne_overlap_cols_cells)
cat("[INFO] GLNE overlap with cells in rows:\n")
print(glne_overlap_rows_cells)

if (glne_overlap_rows_cells > glne_overlap_cols_cells) {
  cat("[INFO] GLNE appears to be cells x genes. No transpose needed.\n")
  GLNE_cell_by_gene <- GLNE_mat
} else {
  cat("[INFO] GLNE appears to be genes x cells. Transposing to cells x genes...\n")
  GLNE_cell_by_gene <- t(GLNE_mat)
}

common_cells_final <- intersect(rownames(raw_hvg_cell_by_gene), rownames(GLNE_cell_by_gene))
common_genes_final <- intersect(colnames(raw_hvg_cell_by_gene), colnames(GLNE_cell_by_gene))

cat("[INFO] Common cells between HVG expression and GLNE:\n")
print(length(common_cells_final))
cat("[INFO] Common genes between HVG expression and GLNE:\n")
print(length(common_genes_final))

if (length(common_cells_final) == 0) stop("[ERROR] No common cells between HVG expression and GLNE.")
if (length(common_genes_final) == 0) stop("[ERROR] No common genes between HVG expression and GLNE.")

raw_hvg_cell_by_gene <- raw_hvg_cell_by_gene[common_cells_final, common_genes_final, drop = FALSE]
scaled_hvg_cell_by_gene <- scaled_hvg_cell_by_gene[common_cells_final, common_genes_final, drop = FALSE]
GLNE_cell_by_gene <- GLNE_cell_by_gene[common_cells_final, common_genes_final, drop = FALSE]

cat("[INFO] Final raw HVG matrix, cells x genes:\n")
print(dim(raw_hvg_cell_by_gene))
cat("[INFO] Final scaled HVG matrix, cells x genes:\n")
print(dim(scaled_hvg_cell_by_gene))
cat("[INFO] Final GLNE matrix, cells x genes:\n")
print(dim(GLNE_cell_by_gene))

final_raw_hvg_path <- file.path(out_dir, "final_raw_HVG_counts_cell_by_gene.csv")
final_scaled_hvg_path <- file.path(out_dir, "final_scaled_HVG_expr_cell_by_gene.csv")
final_glne_path <- file.path(out_dir, "final_GLNE_cell_by_gene.csv")

write_matrix_with_id(raw_hvg_cell_by_gene, final_raw_hvg_path, id_col = "Cell")
write_matrix_with_id(scaled_hvg_cell_by_gene, final_scaled_hvg_path, id_col = "Cell")
write_matrix_with_id(GLNE_cell_by_gene, final_glne_path, id_col = "Cell")

cat("[OK] Final raw HVG expression matrix saved:\n", final_raw_hvg_path, "\n", sep = "")
cat("[OK] Final scaled HVG expression matrix saved:\n", final_scaled_hvg_path, "\n", sep = "")
cat("[OK] Final GLNE matrix saved:\n", final_glne_path, "\n", sep = "")


raw_hvg_minmax <- minmax_scale_columns(raw_hvg_cell_by_gene)
GLNE_minmax <- minmax_scale_columns(GLNE_cell_by_gene)

raw_hvg_minmax_path <- file.path(out_dir, "final_raw_HVG_counts_MinMax_cell_by_gene.csv")
GLNE_minmax_path <- file.path(out_dir, "final_GLNE_MinMax_cell_by_gene.csv")

write_matrix_with_id(raw_hvg_minmax, raw_hvg_minmax_path, id_col = "Cell")
write_matrix_with_id(GLNE_minmax, GLNE_minmax_path, id_col = "Cell")

cat("[OK] Final MinMax raw HVG matrix saved:\n", raw_hvg_minmax_path, "\n", sep = "")
cat("[OK] Final MinMax GLNE matrix saved:\n", GLNE_minmax_path, "\n", sep = "")


metadata_final <- seurat_obj@meta.data
metadata_final <- metadata_final[common_cells_final, , drop = FALSE]

metadata_final_path <- file.path(out_dir, "metadata_final_aligned_after_QC.csv")
metadata_final_out <- cbind(
  Cell = rownames(metadata_final),
  metadata_final[, setdiff(colnames(metadata_final), "Cell"), drop = FALSE]
)
fwrite(metadata_final_out, file = metadata_final_path)

cell_label_final_path <- file.path(out_dir, "cell_labels_after_QC.csv")
cell_label_final_df <- data.frame(
  Cell = rownames(metadata_final),
  Label = metadata_final[[label_col]],
  stringsAsFactors = FALSE
)
fwrite(cell_label_final_df, file = cell_label_final_path)

seurat_rds_path <- file.path(out_dir, "GSE67835_Seurat_CellEnergy_processed_no_regression.rds")
saveRDS(seurat_obj, file = seurat_rds_path)

summary_df <- data.frame(
  item = c(
    "raw_genes_before_qc",
    "raw_cells_before_qc",
    "genes_after_qc",
    "cells_after_qc",
    "hvg_number",
    "final_common_cells",
    "final_common_genes",
    "label_column"
  ),
  value = c(
    nrow(counts),
    ncol(counts),
    nrow(seurat_obj),
    ncol(seurat_obj),
    length(hvg_genes),
    nrow(raw_hvg_cell_by_gene),
    ncol(raw_hvg_cell_by_gene),
    label_col
  )
)
summary_path <- file.path(out_dir, "pipeline_summary.csv")
fwrite(summary_df, file = summary_path)

cat("[OK] Final metadata saved:\n", metadata_final_path, "\n", sep = "")
cat("[OK] Final cell labels saved:\n", cell_label_final_path, "\n", sep = "")
cat("[OK] Seurat object saved:\n", seurat_rds_path, "\n", sep = "")
cat("[OK] Pipeline summary saved:\n", summary_path, "\n", sep = "")

cat("\n[DONE] GSE67835 full pipeline finished successfully.\n")
cat("[DONE] No linear regression was performed.\n\n")
cat("Main outputs:\n")
cat("1. Cell labels before QC:\n", cell_label_path, "\n", sep = "")
cat("2. Cell labels after QC:\n", cell_label_final_path, "\n", sep = "")
cat("3. View 1 for downstream model:\n", raw_hvg_minmax_path, "\n", sep = "")
cat("4. View 2 for downstream model:\n", GLNE_minmax_path, "\n", sep = "")
cat("5. Final raw HVG expression:\n", final_raw_hvg_path, "\n", sep = "")
cat("6. Final GLNE matrix:\n", final_glne_path, "\n", sep = "")
cat("7. Final aligned metadata:\n", metadata_final_path, "\n", sep = "")


[INFO] Expression matrix path:
/home/zhanghang/GSE67835/GSE67835_merged_out/GSE67835_expression_cell_by_gene_raw_counts.csv
[INFO] SOFT metadata path:
/home/zhanghang/GSE67835/GSE67835_merged_out/GSE67835_family.soft
[INFO] Output directory:
/home/zhanghang/GSE67835/Seurat_CellEnergy_full_pipeline

[INFO] Raw expression table dimension from CSV:
[1]   466 22089
[INFO] First columns:
 [1] "V1"          "1/2-SBSRNA4" "A1BG"        "A1BG-AS1"    "A1CF"       
 [6] "A2LD1"       "A2M"         "A2ML1"       "A2MP1"       "A4GALT"     
[INFO] Cell ID column used in expression matrix:
[1] "V1"
[INFO] Removing HTSeq summary features, not real genes:
[1] "no_feature"           "ambiguous"            "alignment_not_unique"
[INFO] Parsed SOFT metadata dimension:
[1] 466  12
[INFO] Parsed metadata columns:
 [1] "soft_sample"            "title"                  "GSM"                   
 [4] "Cell"                   "source_name"            "tissue"                
 [7] "cell_type"              "age

Normalizing layer: counts

Finding variable features for layer counts



[INFO] Number of HVGs:
[1] 2000
[OK] HVG list saved:
/home/zhanghang/GSE67835/Seurat_CellEnergy_full_pipeline/HVG_2000_gene_list.csv


Centering and scaling data matrix



[INFO] Raw HVG counts, genes x cells:
[1] 2000  466
[INFO] Normalized HVG expression, genes x cells:
[1] 2000  466
[INFO] Scaled HVG expression, genes x cells:
[1] 2000  466
[OK] HVG matrices saved.
[INFO] Running CellEnergy calcGEn...


2026-06-11 19:21:54.052438: Starting calculation.

2026-06-11 19:22:00.587668: Complete GLNE and cell Energy calculation.



[INFO] scEnergy preview:
           scEnergy  scEn_Nor
GSM1657871   -28441 0.9634753
GSM1657872   -26106 0.9665115
GSM1657873   -12561 0.9841237
GSM1657874   -15576 0.9802033
GSM1657875   -41616 0.9463442
GSM1657876   -44551 0.9425279
[INFO] GLNE dimension before orientation check:
[1] 2000  466
[OK] Single-cell energy scores saved:
/home/zhanghang/GSE67835/Seurat_CellEnergy_full_pipeline/Energy_scEn.csv
[OK] GLNE gene x cell matrix saved:
/home/zhanghang/GSE67835/Seurat_CellEnergy_full_pipeline/GLNE_gene_by_cell.csv
[INFO] GLNE overlap with cells in columns:
[1] 466
[INFO] GLNE overlap with cells in rows:
[1] 0
[INFO] GLNE appears to be genes x cells. Transposing to cells x genes...
[INFO] Common cells between HVG expression and GLNE:
[1] 466
[INFO] Common genes between HVG expression and GLNE:
[1] 2000
[INFO] Final raw HVG matrix, cells x genes:
[1]  466 2000
[INFO] Final scaled HVG matrix, cells x genes:
[1]  466 2000
[INFO] Final GLNE matrix, cells x genes:
[1]  466 2000
[OK] Final